# Prompt template

In [11]:
PROMPT_TEMPLATE = """
You are an expert in Causal Inference. Your task is to determine the causal direction between two variables based on their context and metadata.

Dataset Context: {context}
Variable X: {var_x_desc}
Variable Y: {var_y_desc}

Based on physical laws, common sense, and scientific facts, select the most plausible causal direction:
- X -> Y (X causes Y)
- Y -> X (Y causes X)
- Independent (No direct causal relationship)

Strict Output Format (DO NOT use any markdown, do not use double asterisks ** anywhere):
Direction: [Your choice: X -> Y, Y -> X, or Independent]
Reason: [Provide a brief explanation in 1-2 sentences]
"""

# Ten pairs of Tuebingen

In [12]:
TUEBINGEN_PAIRS = [
    {
        "pair_id": "0001",
        "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
        "var_x":"altitude",
        "var_y":"temperature (average over 1961-1990)",
        "ground_truth": "X -> Y"

    },
    {
            "pair_id": "0002",
            "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
            "var_x":"altitude",
            "var_y":"precipitation (yearly value averaged over 1961-1990)",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0003",
            "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
            "var_x":"longitude",
            "var_y":"temperature (averaged over 1961-1990)",
            "ground_truth": "X -> Y"
    
    },
    {
            "pair_id": "0004",
            "context": "DWD data (Deutscher Wetterdienst) data was taken at 349 stations",
            "var_x":"altitude",
            "var_y":"sunshine (yearly value averaged over 1961-1990)",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0005",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Length",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0006",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Shell weight",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0007",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Diameter",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0008",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Height",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0009",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Whole weight",
            "ground_truth": "X -> Y"
    },
    {
            "pair_id": "0010",
            "context": "Abalone data",
            "var_x":"Rings",
            "var_y":"Shucked weight",
            "ground_truth": "X -> Y"
    }
]

# Run LLM for causal predictions

In [ ]:
import csv
import ollama

csv_data = []
for pair in TUEBINGEN_PAIRS :
    prompt = PROMPT_TEMPLATE.format(
        context=pair["context"],
        var_x_desc=pair["var_x"],
        var_y_desc=pair["var_y"])

    response = ollama.chat(
        model='llama3.1:8B', 
        messages=[
            {
                'role': 'user',
                'content': prompt,
            },
        ],
        options={ # option to fix the randomization each run
            "temperature": 0,
            "top_p": 1,
            "top_k": 1,
            "seed": 42,
        },
        stream=False  # Enables real-time streaming output
    )
    output = response['message']['content']

    predicted_direction = "N/A"
    reason = "N/A"
    print(output)
    for line in output.split('\n'):
        if line.startswith("Direction:"):
            predicted_direction = line.replace("Direction:", "").strip()
        elif line.startswith("Reason:"):
            reason = line.replace("Reason:", "").strip()

    correctness = "False";
    if pair["ground_truth"] in predicted_direction or predicted_direction in pair["ground_truth"]:
        correctness = "True";
    
    csv_data.append({
        "Pair_ID": pair["pair_id"],
        "Variable_X": pair["var_x"],
        "Variable_Y": pair["var_y"],
        "Predicted_Direction": predicted_direction,
        "Ground_Truth": pair["ground_truth"],
        "Correctness": correctness,
        "Reason": reason
    })

csv_file_name = "../output/causal_predictions.csv"
headers = ["Pair_ID", "Variable_X", "Variable_Y", "Ground_Truth", "Predicted_Direction", "Correctness", "Reason"]

with open(csv_file_name, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(csv_data)

print(f"Finish write to file: {csv_file_name}")

Direction: Y -> X
Reason: Temperature is generally influenced by altitude, as higher altitudes tend to have lower temperatures due to the decrease in atmospheric pressure and the resulting cooling effect. This is a fundamental physical principle that governs the behavior of the atmosphere.
Direction: X -> Y
Reason: Physical laws governing atmospheric conditions suggest that higher altitudes are associated with lower air pressure and temperature, which in turn can lead to increased precipitation. This relationship is well-documented in meteorological literature, supporting the direction from altitude (X) to precipitation (Y).
Direction: X -> Y
Reason: The longitude of a weather station is determined at the time of its establishment and does not change over time. Therefore, it is more plausible that the spatial location (longitude) influences the temperature readings at a given station, rather than the temperature influencing the station's geographical location.
Direction: X -> Y
Reason: